# 03 — Node2Vec Ingredient Embeddings

## Goal
Learn ingredient embeddings from the frozen ingredient co-occurrence graph.
Embeddings should reflect **functional/contextual role similarity** (useful for substitution),
not just raw co-occurrence frequency.

## Frozen inputs (read-only)
- data/processed/ingredient_cooccurrence_edges.csv
  - columns: ingredient_a, ingredient_b, weight
- data/processed/recipe_ingredients.csv (not used directly in this notebook)

## Iteration 0 (baseline hyperparameters)
We start from literature-aligned defaults and a mild DFS bias to encourage role similarity.

- dimension (d): 128
- walk length (l): 60
- walks per node (r): 20
- window size (k): 10
- return parameter (p): 1
- in-out parameter (q): 2
- random seed: 42

## Evidence we will record each iteration
For a fixed probe list of ingredients, we will record:
- Top-10 nearest neighbours (cosine similarity)
- Short notes: what improved / what got worse / next change

We will only change **one parameter at a time** between iterations.


In [8]:
# -----------------------------------------------------------
# A.0 — Load the frozen ingredient co-occurrence edge list
# -----------------------------------------------------------

import pandas as pd  # pandas lets us load and inspect tabular data (CSV files)

# Path to the frozen edge list produced earlier (treat this file as read-only)
EDGES_PATH = "../data/processed/ingredient_cooccurrence_edges.csv"

# Load the CSV into a DataFrame (a table-like structure: rows + columns)
edges_df = pd.read_csv(EDGES_PATH)

# Quick sanity checks / inspection (we want to confirm the schema is as expected)
print("Loaded edges from:", EDGES_PATH)                 # show which file we loaded
print("Number of edges (rows):", len(edges_df))         # how many co-occurrence pairs
print("Columns:", list(edges_df.columns))               # confirm column names

# Show the first few rows so we can visually verify the format
edges_df.head()


Loaded edges from: ../data/processed/ingredient_cooccurrence_edges.csv
Number of edges (rows): 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']


,ingredient_a,ingredient_b,weight
0,egg,flour,27623
1,egg,milk,16380
2,flour,milk,16133
3,flour,powder,15622
4,egg,powder,13475


In [9]:
# -----------------------------------------------------------
# A.1 — Build the weighted undirected ingredient graph
# -----------------------------------------------------------

import networkx as nx  # NetworkX gives us a graph object we can run Node2Vec on

# Create an empty undirected graph (because ingredient co-occurrence is unordered)
G = nx.Graph()

# Add all edges in one go, storing the co-occurrence count as the edge attribute "weight"
# Each row becomes an edge: ingredient_a -- ingredient_b with weight=<count>
G.add_weighted_edges_from(
    edges_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

# Quick visibility checks (not correctness checks)
print("Graph built.")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

# Example: show the weight of the strongest edge we saw in the CSV preview
print("Weight(egg, flour):", G["egg"]["flour"]["weight"])


Graph built.
Number of nodes: 2698
Number of edges: 419913
Weight(egg, flour): 27623


In [10]:
# -----------------------------------------------------------
# B.0 — Train Node2Vec (Iteration 0 baseline)
# -----------------------------------------------------------

from node2vec import Node2Vec  # Node2Vec generates biased random walks + trains embeddings

# 1) Prepare random walks + training setup
node2vec = Node2Vec(
    G,                      # our ingredient graph
    dimensions=128,         # d: size of each embedding vector
    walk_length=60,         # l: steps per walk
    num_walks=20,           # r: walks started from each node
    p=1,                    # return bias (baseline)
    q=2,                    # in-out bias (mild DFS → role similarity)
    weight_key="weight",    # use edge weights when choosing next step
    workers=4,              # how many CPU cores to use
    seed=42                 # reproducibility
)

# 2) Train skip-gram (Word2Vec) on the walks to learn embeddings
model = node2vec.fit(
    window=10,              # k: context window size
    min_count=1,            # keep all ingredients (even rare ones)
    batch_words=4           # training batch size
)

print("Iteration 0 training complete.")


Computing transition probabilities: 100%|██████████| 2698/2698 [37:22<00:00,  1.20it/s]    
/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Generating walks (CPU: 4): 100%|██████████| 5/5 [00:37<00:00,  7.45s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception i

Iteration 0 training complete.


In [11]:
# -----------------------------------------------------------
# B.1 — Save Iteration 0 embeddings (so we don’t retrain)
# -----------------------------------------------------------

from pathlib import Path

OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save in Word2Vec text format (easy to load later)
emb_path = OUT_DIR / "ingredient_node2vec_iter0.txt"
model.wv.save_word2vec_format(str(emb_path))

print("Saved embeddings to:", emb_path)
print("Number of embeddings saved:", len(model.wv))
print("Embedding dimension:", model.vector_size)
# -----------------------------------------------------------
# B.1 — Save Iteration 0 embeddings (so we don’t retrain)
# -----------------------------------------------------------

from pathlib import Path

OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save in Word2Vec text format (easy to load later)
emb_path = OUT_DIR / "ingredient_node2vec_iter0.txt"
model.wv.save_word2vec_format(str(emb_path))

print("Saved embeddings to:", emb_path)
print("Number of embeddings saved:", len(model.wv))
print("Embedding dimension:", model.vector_size)


Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter0.txt
Number of embeddings saved: 2698
Embedding dimension: 128
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter0.txt
Number of embeddings saved: 2698
Embedding dimension: 128


In [12]:
# -----------------------------------------------------------
# C.0 — Iteration 0 evidence: nearest-neighbour sanity checks
# -----------------------------------------------------------

# A fixed probe list (keep this constant across all iterations)
# We choose ingredients that are common, substitutable, and cover different roles.
probe_ingredients = [
    "butter", "olive oil", "vegetable oil",
    "milk", "yogurt", "cream",
    "flour", "bread flour",
    "sugar", "brown sugar", "honey",
    "egg",
    "garlic", "onion",
    "chicken", "beef",
    "rice", "pasta"
]

# Some ingredients may not exist *exactly* with these spellings in your vocabulary.
# This helper checks which probes are in-vocab vs missing.
vocab = set(model.wv.key_to_index.keys())
present = [x for x in probe_ingredients if x in vocab]
missing = [x for x in probe_ingredients if x not in vocab]

print("Probe ingredients present:", present)
print("Probe ingredients missing:", missing)

# Print top-10 neighbours for each present probe
TOPN = 10
for ing in present:
    print("\n" + "=" * 60)
    print(f"Probe: {ing}")
    neighbours = model.wv.most_similar(ing, topn=TOPN)
    for rank, (nbr, sim) in enumerate(neighbours, start=1):
        print(f"{rank:02d}. {nbr:<25}  cosine={sim:.3f}")


Probe ingredients present: ['milk', 'yogurt', 'cream', 'flour', 'bread flour', 'honey', 'egg', 'garlic', 'chicken', 'beef', 'rice', 'pasta']
Probe ingredients missing: ['butter', 'olive oil', 'vegetable oil', 'sugar', 'brown sugar', 'onion']

Probe: milk
01. egg                        cosine=0.770
02. english walnut             cosine=0.758
03. coconut pecan frosting     cosine=0.749
04. flour                      cosine=0.749
05. peppermint patty           cosine=0.741
06. milky way bar              cosine=0.738
07. margarine                  cosine=0.737
08. special k cereal           cosine=0.737
09. cream                      cosine=0.721
10. real vanilla               cosine=0.714

Probe: yogurt
01. soy yogurt                 cosine=0.624
02. sumaq                      cosine=0.609
03. vegetarian sausage         cosine=0.606
04. mango pulp                 cosine=0.604
05. lamb rib chop              cosine=0.599
06. lamb shoulder              cosine=0.596
07. lemon juice           

In [13]:
# -----------------------------------------------------------
# D.0 — Iteration 1: increase DFS bias to reduce hub dominance
# (Only change: q from 2 → 4)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 1 hyperparameters (ONLY q changes) ---
ITER = 1
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
WINDOW = 10
P = 1
Q = 4          # <-- the only intentional change vs Iteration 0
SEED = 42

# 1) Create Node2Vec object (walk generation setup)
node2vec_iter1 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,         # embedding size
    walk_length=WALK_LEN,   # steps per walk
    num_walks=NUM_WALKS,    # walks per node
    p=P,                    # return parameter (unchanged)
    q=Q,                    # in-out parameter (changed)
    weight_key="weight",    # use co-occurrence weights in transitions
    workers=2,              # more stable than 4 on Mac laptops
    seed=SEED               # reproducibility
)

# 2) Train skip-gram model on walks (learn embeddings)
# Setting workers=1 here reduces gensim thread/shutdown noise.
model_iter1 = node2vec_iter1.fit(
    window=WINDOW,          # context window
    min_count=1,            # keep all ingredients
    batch_words=4,          # stable small batches
    workers=1               # stability: avoids noisy "Exception ignored" spam
)

print(f"Iteration {ITER} training complete (p={P}, q={Q}).")

# 3) Save embeddings to disk (new filename so we can compare later)
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter1 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_p{P}_q{Q}.txt"
model_iter1.wv.save_word2vec_format(str(emb_path_iter1))

print("Saved embeddings to:", emb_path_iter1)
print("Number of embeddings saved:", len(model_iter1.wv))
print("Embedding dimension:", model_iter1.vector_size)


Generating walks (CPU: 2): 100%|██████████| 10/10 [01:18<00:00,  7.89s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 1 training complete (p=1, q=4).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter1_p1_q4.txt
Number of embeddings saved: 2698
Embedding dimension: 128


In [14]:
# -----------------------------------------------------------
# D.1 — Iteration 1 evidence: nearest neighbours (same probes)
# -----------------------------------------------------------

# Reuse the same probe list (do NOT change it)
vocab1 = set(model_iter1.wv.key_to_index.keys())
present1 = [x for x in probe_ingredients if x in vocab1]
missing1 = [x for x in probe_ingredients if x not in vocab1]

print("Probe ingredients present:", present1)
print("Probe ingredients missing:", missing1)

TOPN = 10
for ing in present1:
    print("\n" + "=" * 60)
    print(f"Probe: {ing}")
    neighbours = model_iter1.wv.most_similar(ing, topn=TOPN)
    for rank, (nbr, sim) in enumerate(neighbours, start=1):
        print(f"{rank:02d}. {nbr:<25}  cosine={sim:.3f}")


Probe ingredients present: ['milk', 'yogurt', 'cream', 'flour', 'bread flour', 'honey', 'egg', 'garlic', 'chicken', 'beef', 'rice', 'pasta']
Probe ingredients missing: ['butter', 'olive oil', 'vegetable oil', 'sugar', 'brown sugar', 'onion']

Probe: milk
01. egg                        cosine=0.775
02. margarine                  cosine=0.771
03. honey graham cracker       cosine=0.768
04. egg bread                  cosine=0.758
05. puree                      cosine=0.752
06. oreo cookie pie crust      cosine=0.751
07. flour                      cosine=0.748
08. milky way bar              cosine=0.740
09. skor candy bar             cosine=0.737
10. chai tea                   cosine=0.735

Probe: yogurt
01. pomegranate syrup          cosine=0.633
02. smooth cashew butter       cosine=0.624
03. mint sauce                 cosine=0.623
04. lemon juice                cosine=0.620
05. red plum                   cosine=0.616
06. 1 fat buttermilk           cosine=0.597
07. bourbon whiskey       

In [16]:
# -----------------------------------------------------------
# E.1 — Canonical probe list (mapped to actual vocab tokens)
# -----------------------------------------------------------

probe_ingredients_canonical = [
    # fats / oils
    "real butter",
    "pure olive oil",
    "canola oil",              # neutral oil proxy for "vegetable oil"
    
    # dairy
    "milk", "yogurt", "cream",
    
    # baking staples
    "flour", "bread flour",
    "cane sugar",
    "light-brown sugar",
    "honey",
    
    # aromatics
    "garlic",
    "red onion",
    
    # proteins
    "chicken", "beef",
    
    # carbs
    "rice", "pasta"
]

def print_neighbours(model, probe_list, topn=10, title=""):
    vocab = set(model.wv.key_to_index.keys())
    present = [x for x in probe_list if x in vocab]
    missing = [x for x in probe_list if x not in vocab]
    
    if title:
        print("\n" + "#" * 70)
        print(title)
        print("#" * 70)
        
    print("Present probes:", present)
    print("Missing probes:", missing)
    
    for ing in present:
        print("\n" + "=" * 60)
        print(f"Probe: {ing}")
        for i, (nbr, sim) in enumerate(model.wv.most_similar(ing, topn=topn), start=1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

# Run for Iteration 0 model
print_neighbours(model, probe_ingredients_canonical, topn=10, title="Iteration 0 neighbours (canonical probes)")

# Run for Iteration 1 model
print_neighbours(model_iter1, probe_ingredients_canonical, topn=10, title="Iteration 1 neighbours (canonical probes)")



######################################################################
Iteration 0 neighbours (canonical probes)
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. 18 table cream                 cosine=0.630
02. club cracker                   cosine=0.601
03. candy cane                     cosine=0.593
04. ready-made pie crust           cosine=0.585
05. tap water                      cosine=0.570
06. vanilla chip                   cosine=0.562
07. chocolate icing                cosine=0.559
08. cherry jam                     cosine=0.558
09. buttermilk pancake             cosine=0.556
10. egg bread                      cosine=0.555

Probe: pure olive oil
01. hazelnut oil                   cosine=0.686
02. kos

In [17]:
# -----------------------------------------------------------
# F.0 — Iteration 2: reduce context mixing by shrinking window
# (Only change: window from 10 → 5, keep p=1, q=2)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 2 hyperparameters (ONLY window changes vs Iteration 0) ---
ITER = 2
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 2               # back to the baseline exploration setting
WINDOW = 5          # <-- the only intentional change vs Iteration 0
SEED = 42

# 1) Create Node2Vec object (random walk generation setup)
node2vec_iter2 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,         # size of each embedding vector
    walk_length=WALK_LEN,   # steps per walk
    num_walks=NUM_WALKS,    # walks started from each node
    p=P,                    # return parameter
    q=Q,                    # in-out parameter
    weight_key="weight",    # use co-occurrence counts in transitions
    workers=2,              # stable on Mac laptops
    seed=SEED               # reproducibility
)

# 2) Train skip-gram model on the walks (learn embeddings)
# workers=1 here reduces gensim shutdown/thread noise
model_iter2 = node2vec_iter2.fit(
    window=WINDOW,          # <-- only change vs Iteration 0
    min_count=1,            # keep all ingredients
    batch_words=4,          # small stable batches
    workers=1               # stability
)

print(f"Iteration {ITER} training complete (p={P}, q={Q}, window={WINDOW}).")

# 3) Save embeddings to disk (new filename for comparison)
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter2 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_p{P}_q{Q}_w{WINDOW}.txt"
model_iter2.wv.save_word2vec_format(str(emb_path_iter2))

print("Saved embeddings to:", emb_path_iter2)
print("Number of embeddings saved:", len(model_iter2.wv))
print("Embedding dimension:", model_iter2.vector_size)

# -----------------------------------------------------------
# F.1 — Iteration 2 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter2,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 2 neighbours (canonical probes) — window reduced to 5"
)


Generating walks (CPU: 2): 100%|██████████| 10/10 [01:23<00:00,  8.35s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 2 training complete (p=1, q=2, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter2_p1_q2_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 2 neighbours (canonical probes) — window reduced to 5
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. red raspberry preserve         cosine=0.664
02. milk chocolate kisse           cosine=0.646
03. pastry crust                   cosine=0.645
04. phyllo cup                     cosine=0.637
05. milk chocolate candy bar       cosine=0.630
06. ande mint candy                cosine=0.625
07. chocolate cookie               cosine=0.623
0

In [18]:
# -----------------------------------------------------------
# G.0 — Iteration 3: remove edge weights (UNWEIGHTED walks)
# (Only change: do NOT use weight_key)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 3 hyperparameters (IDENTICAL to Iteration 2) ---
ITER = 3
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 2
WINDOW = 5
SEED = 42

# 1) Create Node2Vec object (UNWEIGHTED random walks)
node2vec_iter3 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: weight_key intentionally omitted
    workers=2,
    seed=SEED
)

# 2) Train skip-gram model
model_iter3 = node2vec_iter3.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1
)

print(f"Iteration {ITER} training complete (UNWEIGHTED walks).")

# 3) Save embeddings
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter3 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}.txt"
model_iter3.wv.save_word2vec_format(str(emb_path_iter3))

print("Saved embeddings to:", emb_path_iter3)
print("Number of embeddings saved:", len(model_iter3.wv))
print("Embedding dimension:", model_iter3.vector_size)

# -----------------------------------------------------------
# G.1 — Iteration 3 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter3,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 3 neighbours (canonical probes) — UNWEIGHTED walks"
)

Generating walks (CPU: 2): 100%|██████████| 10/10 [01:25<00:00,  8.57s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 3 training complete (UNWEIGHTED walks).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter3_UNWEIGHTED_p1_q2_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 3 neighbours (canonical probes) — UNWEIGHTED walks
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. preserve                       cosine=0.658
02. non-hydrogenated margarine     cosine=0.641
03. dutch process cocoa            cosine=0.637
04. real vanilla extract           cosine=0.637
05. mint chocolate                 cosine=0.635
06. jumbo egg                      cosine=0.635
07. extra-large egg yolk           cosine=0

In [19]:
# -----------------------------------------------------------
# H.0 — Iteration 4: UNWEIGHTED + stronger DFS bias to push structural equivalence
# (Only change vs Iteration 3: q from 2 → 4)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 4 hyperparameters (ONLY q changes vs Iteration 3) ---
ITER = 4
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 4            # <-- ONLY intentional change vs Iteration 3 (q=2 → q=4)
WINDOW = 5
SEED = 42

# 1) Create Node2Vec object (UNWEIGHTED random walks)
# IMPORTANT: weight_key intentionally omitted to keep walks unweighted.
node2vec_iter4 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    workers=2,              # stable on Mac
    seed=SEED
)

# 2) Train skip-gram model
model_iter4 = node2vec_iter4.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1               # stability: reduces shutdown/thread noise
)

print(f"Iteration {ITER} training complete (UNWEIGHTED, p={P}, q={Q}, window={WINDOW}).")

# 3) Save embeddings
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter4 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}.txt"
model_iter4.wv.save_word2vec_format(str(emb_path_iter4))

print("Saved embeddings to:", emb_path_iter4)
print("Number of embeddings saved:", len(model_iter4.wv))
print("Embedding dimension:", model_iter4.vector_size)

# -----------------------------------------------------------
# H.1 — Iteration 4 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter4,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 4 neighbours (canonical probes) — UNWEIGHTED, q increased to 4"
)


Generating walks (CPU: 2): 100%|██████████| 10/10 [01:23<00:00,  8.40s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 4 training complete (UNWEIGHTED, p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter4_UNWEIGHTED_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 4 neighbours (canonical probes) — UNWEIGHTED, q increased to 4
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. popcorn                        cosine=0.709
02. buttermilk pancake             cosine=0.706
03. dipping chocolate              cosine=0.694
04. milky way bar                  cosine=0.694
05. 1 fat buttermilk               cosine=0.689
06. mint chocolate                 cosine=0.686
07. crisco oil   

In [20]:
# -----------------------------------------------------------
# I.0 — Iteration 5: Association-Strength WEIGHTED graph (same params as Iteration 4)
# (Only change vs Iteration 4: switch graph source to assoc-strength weighted edges)
# -----------------------------------------------------------

import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# Path to the NEW normalised edge list (association strength weights)
EDGES_ASSOC_PATH = "../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv"

# --- Iteration 5 hyperparameters (IDENTICAL to Iteration 4) ---
ITER = 5
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 4
WINDOW = 5
SEED = 42

# 1) Load the association-strength edge list (same columns, different meaning of "weight")
edges_assoc_df = pd.read_csv(EDGES_ASSOC_PATH)

print("Loaded association-strength edges from:", EDGES_ASSOC_PATH)
print("Number of edges (rows):", len(edges_assoc_df))
print("Columns:", list(edges_assoc_df.columns))
display(edges_assoc_df.head())

# 2) Build a NEW graph object so we do NOT overwrite the original G
# This keeps all previous iterations comparable (each one uses its own graph choice).
G_assoc = nx.Graph()

# Add edges with association-strength weights
# Each row becomes: ingredient_a -- ingredient_b with weight=<association_strength>
G_assoc.add_weighted_edges_from(
    edges_assoc_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

print("Association-strength graph built.")
print("Number of nodes:", G_assoc.number_of_nodes())
print("Number of edges:", G_assoc.number_of_edges())

# Optional quick check: show a familiar edge weight if it exists in this graph
# (Not all classic pairs will necessarily appear if your edge list was filtered earlier.)
if G_assoc.has_edge("egg", "flour"):
    print("AssocWeight(egg, flour):", G_assoc["egg"]["flour"]["weight"])
else:
    print("Edge (egg, flour) not found in assoc graph (this can happen if the edge list was filtered).")

# 3) Train Node2Vec on the association-strength WEIGHTED graph
# IMPORTANT: we DO use weight_key here, because the whole point is to test this weighting scheme.
node2vec_iter5 = Node2Vec(
    G_assoc,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",   # <-- use association-strength weights during transitions
    workers=2,
    seed=SEED
)

# 4) Fit skip-gram (Word2Vec) model to learn embeddings
model_iter5 = node2vec_iter5.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1              # stability on Mac
)

print(f"Iteration {ITER} training complete (ASSOC-WEIGHTED, p={P}, q={Q}, window={WINDOW}).")

# 5) Save embeddings (new filename to keep comparisons clean)
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter5 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_ASSOC_WEIGHTED_p{P}_q{Q}_w{WINDOW}.txt"
model_iter5.wv.save_word2vec_format(str(emb_path_iter5))

print("Saved embeddings to:", emb_path_iter5)
print("Number of embeddings saved:", len(model_iter5.wv))
print("Embedding dimension:", model_iter5.vector_size)

# -----------------------------------------------------------
# I.1 — Iteration 5 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter5,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 5 neighbours (canonical probes) — ASSOC-WEIGHTED graph, same params as Iteration 4"
)


Loaded association-strength edges from: ../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv
Number of edges (rows): 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']


,ingredient_a,ingredient_b,weight
0,vegan cream cheese,vegan parmesan cheese,0.010417
1,corn husk,masa harina flour,0.009804
2,capicola,mortadella,0.008791
3,corn chex,wheat chex,0.008222
4,rice chex,wheat chex,0.007477


Association-strength graph built.
Number of nodes: 2698
Number of edges: 419913
AssocWeight(egg, flour): 1.0892050059415712e-05


Generating walks (CPU: 2): 100%|██████████| 10/10 [00:39<00:00,  4.00s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Iteration 5 training complete (ASSOC-WEIGHTED, p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter5_ASSOC_WEIGHTED_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 5 neighbours (canonical probes) — ASSOC-WEIGHTED graph, same params as Iteration 4
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. real vanilla extract           cosine=0.454
02. mountain dew soda              cosine=0.453
03. butter recipe cake             cosine=0.429
04. fig preserve                   cosine=0.401
05. unbaked pie crust              cosine=0.395
06. real vanilla                   co

In [21]:
# -----------------------------------------------------------
# J.0 — Diagnose Iteration 5 + choose the next best move
# -----------------------------------------------------------
# Why Iteration 5 looks worse:
# Association-strength weights can make the graph extremely "spiky":
# - rare–rare pairs get disproportionately high weights
# - random walks over-weight niche edges and under-sample common functional neighbours
# Result: neighbours become quirky / noisy (e.g., "real butter" ~ "mountain dew soda")

import numpy as np
import pandas as pd

def summarize_weight_distribution(edges_df, name="edges"):
    w = edges_df["weight"].astype(float).values
    print(f"\n=== {name}: weight distribution summary ===")
    print("count:", len(w))
    for p in [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.5, 99.9, 100]:
        print(f"p{p:>5}: {np.percentile(w, p):.12g}")
    print("min:", w.min(), "max:", w.max())
    print("mean:", w.mean(), "median:", np.median(w))

# Compare raw-count weights vs association-strength weights
summarize_weight_distribution(edges_df, name="RAW co-occurrence (count weights)")
summarize_weight_distribution(edges_assoc_df, name="ASSOC strength weights")

# Show the top assoc-strength edges: these are often rare/rare pairs with small counts
# which can dominate walks if used directly as transition weights.
print("\nTop 15 assoc-strength edges (often rare/rare pairs):")
display(
    edges_assoc_df.sort_values("weight", ascending=False)
                 .head(15)
)

# -----------------------------------------------------------
# J.1 — Recommended next step (best outcome): keep assoc-strength for PRUNING,
# but use log-count for WALK transitions
# -----------------------------------------------------------
# This is exactly the strategy you validated earlier:
# - assoc-strength chooses which edges survive (reduces hub dominance)
# - log1p(count) provides stable transition weights (reduces rare-edge domination)

# Inputs you already have:
# - edges_assoc_df: assoc-strength weights for ALL edges
# - edges_df: raw counts (weight column == count) for ALL edges
# We will:
# 1) merge counts into assoc table
# 2) for each node, keep top-K by assoc-strength
# 3) collapse to undirected set
# 4) set final walk weights = log1p(count)
# 5) train Node2Vec on the resulting pruned+logweighted graph

TOPK = 50  # keep top 50 strongest assoc edges per node (good default)

# 1) Merge raw counts into assoc edges
raw_counts = edges_df.rename(columns={"weight": "count"})
assoc = edges_assoc_df.merge(raw_counts, on=["ingredient_a", "ingredient_b"], how="left")
print("Missing counts after merge (should be 0):", assoc["count"].isna().sum())

# 2) Build directed top-K (each node keeps its best neighbours by assoc strength)
# Make a directed view by duplicating each undirected edge both directions.
a_to_b = assoc[["ingredient_a", "ingredient_b", "weight", "count"]].copy()
b_to_a = assoc.rename(columns={"ingredient_a": "ingredient_b", "ingredient_b": "ingredient_a"})[
    ["ingredient_a", "ingredient_b", "weight", "count"]
].copy()

directed = pd.concat([a_to_b, b_to_a], ignore_index=True)

# For each source node, keep top-K outgoing edges by assoc-strength weight
directed_topk = (
    directed.sort_values(["ingredient_a", "weight"], ascending=[True, False])
            .groupby("ingredient_a", as_index=False)
            .head(TOPK)
)

print("Directed edges after top-K pruning:", len(directed_topk))

# 3) Collapse to undirected edge set (canonical ordering)
def canon_pair(row):
    a, b = row["ingredient_a"], row["ingredient_b"]
    return (a, b) if a < b else (b, a)

directed_topk["a_canon"], directed_topk["b_canon"] = zip(*directed_topk.apply(canon_pair, axis=1))

undirected_pruned = (
    directed_topk.groupby(["a_canon", "b_canon"], as_index=False)
                 .agg({"count": "max"})   # if edge appears twice, counts are same; max is safe
                 .rename(columns={"a_canon": "ingredient_a", "b_canon": "ingredient_b"})
)

print("Undirected edges after pruning:", len(undirected_pruned))

# 4) Define walk weights as log1p(count)
undirected_pruned["weight"] = np.log1p(undirected_pruned["count"].astype(float))

# Quick sanity: top weights should be common pairs, not weird rare ones
print("\nTop 10 edges after pruning (by log-count walk weight):")
display(
    undirected_pruned.sort_values("weight", ascending=False)
                    .head(10)[["ingredient_a", "ingredient_b", "weight", "count"]]
)

# 5) Save pruned edge list (so it becomes a clean frozen input for embeddings)
PRUNED_EDGES_PATH = "../data/processed/ingredient_edges_pruned_by_assoc_top50_logcount.csv"
undirected_pruned[["ingredient_a", "ingredient_b", "weight"]].to_csv(PRUNED_EDGES_PATH, index=False)
print("Saved pruned edge list to:", PRUNED_EDGES_PATH)
print("Rows saved:", len(undirected_pruned))

# -----------------------------------------------------------
# J.2 — Iteration 6: Node2Vec on PRUNED graph (assoc-pruned + logcount-walk)
# -----------------------------------------------------------

import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 6 hyperparameters ---
ITER = 6
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 4          # keep the stronger DFS bias that helped before
WINDOW = 5
SEED = 42

# Build graph from the new pruned edge list
G_pruned = nx.Graph()
G_pruned.add_weighted_edges_from(
    undirected_pruned[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

print("\nPruned graph built.")
print("Nodes:", G_pruned.number_of_nodes())
print("Edges:", G_pruned.number_of_edges())

# Train Node2Vec using the pruned + log-count weights
node2vec_iter6 = Node2Vec(
    G_pruned,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",   # use log-count weights for transitions
    workers=2,
    seed=SEED
)

model_iter6 = node2vec_iter6.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1
)

print(f"Iteration {ITER} training complete (ASSOC-PRUNED + LOGCOUNT-WEIGHTED, p={P}, q={Q}, window={WINDOW}).")

# Save embeddings
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter6 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_ASSOC_PRUNED_top{TOPK}_LOGCOUNT_p{P}_q{Q}_w{WINDOW}.txt"
model_iter6.wv.save_word2vec_format(str(emb_path_iter6))

print("Saved embeddings to:", emb_path_iter6)
print("Number of embeddings saved:", len(model_iter6.wv))
print("Embedding dimension:", model_iter6.vector_size)

# Evidence: neighbours for canonical probes
print_neighbours(
    model_iter6,
    probe_ingredients_canonical,
    topn=10,
    title=f"Iteration {ITER} neighbours — ASSOC-PRUNED(top{TOPK}) + LOGCOUNT walk weights"
)



=== RAW co-occurrence (count weights): weight distribution summary ===
count: 419913
p    0: 1
p    1: 1
p    5: 1
p   10: 1
p   25: 1
p   50: 2
p   75: 5
p   90: 16
p   95: 36
p   99: 188
p 99.5: 348
p 99.9: 1184.176
p  100: 27623
min: 1.0 max: 27623.0
mean: 12.90589241104705 median: 2.0

=== ASSOC strength weights: weight distribution summary ===
count: 419913
p    0: 6.1058288263e-09
p    1: 3.66700800355e-07
p    5: 1.04835611668e-06
p   10: 1.76038982151e-06
p   25: 4.00775902147e-06
p   50: 9.9025588212e-06
p   75: 2.83687943262e-05
p   90: 8.89110179658e-05
p   95: 0.000180310133429
p   99: 0.000651958484251
p 99.5: 0.000965250965251
p 99.9: 0.0018286659577
p  100: 0.0104166666667
min: 6.1058288263005205e-09 max: 0.0104166666666666
mean: 4.4785437698621914e-05 median: 9.9025588211994e-06

Top 15 assoc-strength edges (often rare/rare pairs):


,ingredient_a,ingredient_b,weight
0,vegan cream cheese,vegan parmesan cheese,0.010417
1,corn husk,masa harina flour,0.009804
2,capicola,mortadella,0.008791
3,corn chex,wheat chex,0.008222
4,rice chex,wheat chex,0.007477
5,channa dal,mung dal,0.007143
6,butter pecan cake,coconut pecan frosting,0.006803
7,paper cup,wooden popsicle stick,0.006579
8,nori,sushi rice,0.006531
9,hing,mung dal,0.006522


Missing counts after merge (should be 0): 0
Directed edges after top-K pruning: 134648
Undirected edges after pruning: 81362

Top 10 edges after pruning (by log-count walk weight):


,ingredient_a,ingredient_b,weight,count
47612,flour,powder,9.656499,15622
43828,egg,powder,9.508666,13475
47623,flour,soda,9.436519,12537
43839,egg,soda,9.311994,11069
74150,powder,soda,8.818038,6754
32698,cinnamon,nutmeg,8.513787,4982
79192,soda,vanilla,8.398184,4438
23252,carrot,celery,8.222016,3721
9674,basil,oregano,8.143517,3440
29175,chili powder,cumin,8.080856,3231


Saved pruned edge list to: ../data/processed/ingredient_edges_pruned_by_assoc_top50_logcount.csv
Rows saved: 81362

Pruned graph built.
Nodes: 2698
Edges: 81362


Generating walks (CPU: 2): 100%|██████████| 10/10 [00:05<00:00,  1.80it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Iteration 6 training complete (ASSOC-PRUNED + LOGCOUNT-WEIGHTED, p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter6_ASSOC_PRUNED_top50_LOGCOUNT_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 6 neighbours — ASSOC-PRUNED(top50) + LOGCOUNT walk weights
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. butter recipe cake             cosine=0.413
02. cream cheese frosting          cosine=0.407
03. chervil                        cosine=0.406
04. real maple syrup               cosine=0.399
05. frosting                       cosine=0.397
06. cream of potato soup      

In [22]:
# ===========================================================
# NEW: Node2Vec on the INGREDIENT–RECIPE BIPARTITE GRAPH
# (2-step ingredient-only walks: ingredient -> recipe -> ingredient)
# ===========================================================

import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# -----------------------------------------------------------
# A.0 — Load the frozen bipartite edge list
# -----------------------------------------------------------

BIP_EDGES_PATH = "../data/processed/ingredient_recipe_bipartite_edges.csv"

bip_df = pd.read_csv(BIP_EDGES_PATH)

print("Loaded bipartite edges from:", BIP_EDGES_PATH)
print("Rows (edges):", len(bip_df))
print("Columns:", list(bip_df.columns))
display(bip_df.head())

# Safety checks (match the design you already validated)
assert set(bip_df.columns) == {"node_u", "node_v", "weight"}, "Unexpected columns."
assert (bip_df["node_u"].str.startswith("recipe::")).all(), "node_u must be recipe::..."
assert (~bip_df["node_v"].str.startswith("recipe::")).all(), "node_v must be ingredient tokens."
assert (bip_df["weight"] > 0).all(), "Found non-positive weights."

# -----------------------------------------------------------
# A.1 — Build a bipartite NetworkX graph
# -----------------------------------------------------------

B = nx.Graph()

# Add edges: recipe node <-> ingredient node (weight is usually 1)
B.add_weighted_edges_from(
    bip_df[["node_u", "node_v", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

# Split nodes
recipe_nodes = [n for n in B.nodes if str(n).startswith("recipe::")]
ingredient_nodes = [n for n in B.nodes if not str(n).startswith("recipe::")]

print("\nBipartite graph built.")
print("Recipe nodes:", len(recipe_nodes))
print("Ingredient nodes:", len(ingredient_nodes))
print("Total nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

# -----------------------------------------------------------
# A.2 — (IMPORTANT) Build a 2-step INGREDIENT projection for Node2Vec
# Why: Node2Vec on the raw bipartite graph mixes node types; for substitution we want
# ingredient-ingredient similarity from shared recipe contexts.
#
# We construct a new ingredient graph where:
# - nodes are ingredients
# - edge weight between i and j counts how many recipes contain both (bipartite 2-hop)
# - we do NOT create recipe nodes in this final graph
# -----------------------------------------------------------

from itertools import combinations
from collections import Counter

# Build: recipe -> set(ingredients)
# (Using sets ensures a recipe contributes at most 1 to any ingredient pair.)
recipe_to_ings = {}
for r in recipe_nodes:
    nbrs = B.neighbors(r)
    recipe_to_ings[r] = set(nbrs)

# Count shared-recipe co-membership via 2-hop paths
pair_counts = Counter()
for r, ings in recipe_to_ings.items():
    if len(ings) < 2:
        continue
    # unordered pairs of ingredients in this recipe
    for a, b in combinations(sorted(ings), 2):
        pair_counts[(a, b)] += 1

print("\n2-step ingredient projection built.")
print("Unique ingredient-ingredient edges:", len(pair_counts))

# Convert to DataFrame (counts)
proj_df = pd.DataFrame(
    [{"ingredient_a": a, "ingredient_b": b, "count": c} for (a, b), c in pair_counts.items()]
)

# -----------------------------------------------------------
# A.3 — Degree-normalised weighting for the projected graph
# Motivation: hub ingredients dominate; we normalise by ingredient degrees in the
# bipartite graph (how many recipes each ingredient appears in).
#
# We use a simple bipartite-degree normalisation:
#   w_ij = c_ij / sqrt(deg_i * deg_j)
# where deg_i is ingredient degree in the bipartite graph (recipes containing i).
# This is gentler than assoc-strength and usually keeps common functional edges alive.
# -----------------------------------------------------------

# ingredient degree in bipartite graph = number of recipes containing ingredient
deg_ing = dict(B.degree(ingredient_nodes))

proj_df["deg_a"] = proj_df["ingredient_a"].map(deg_ing).astype(float)
proj_df["deg_b"] = proj_df["ingredient_b"].map(deg_ing).astype(float)

# defensive (should not happen)
proj_df = proj_df.dropna(subset=["deg_a", "deg_b"])

proj_df["weight"] = proj_df["count"] / np.sqrt(proj_df["deg_a"] * proj_df["deg_b"])

# Sort for inspection
proj_df = proj_df.sort_values("weight", ascending=False).reset_index(drop=True)

print("\nTop 10 edges after bipartite-degree normalisation:")
display(proj_df.head(10)[["ingredient_a", "ingredient_b", "count", "deg_a", "deg_b", "weight"]])

# Save this as a frozen input for embeddings (recommended)
OUT_DIR = Path("../data/processed")
PROJ_EDGES_PATH = OUT_DIR / "ingredient_edges_from_bipartite_deg_norm.csv"
proj_df[["ingredient_a", "ingredient_b", "weight"]].to_csv(PROJ_EDGES_PATH, index=False)

print("\nSaved projected ingredient edge list to:", PROJ_EDGES_PATH)
print("Rows saved:", len(proj_df))

# -----------------------------------------------------------
# B.0 — Train Node2Vec on the projected ingredient graph
# Keep your best-performing Node2Vec settings (unweighted/DFS-biased style),
# but we DO use weight_key here because weights are now “hub-mitigated” and smooth.
# -----------------------------------------------------------

G_bip_proj = nx.Graph()
G_bip_proj.add_weighted_edges_from(
    proj_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

print("\nProjected graph ready for Node2Vec.")
print("Nodes:", G_bip_proj.number_of_nodes())
print("Edges:", G_bip_proj.number_of_edges())

# --- Iteration label for this new graph family ---
ITER = "BIPPROJ_DEGNORM"

DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
WINDOW = 5
P = 1
Q = 4
SEED = 42

node2vec_bip = Node2Vec(
    G_bip_proj,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",
    workers=2,
    seed=SEED
)

model_bip = node2vec_bip.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1
)

print(f"\nTraining complete: {ITER} (p={P}, q={Q}, window={WINDOW}).")

# -----------------------------------------------------------
# B.1 — Save embeddings
# -----------------------------------------------------------

EMB_DIR = Path("../data/processed/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

emb_path = EMB_DIR / f"ingredient_node2vec_{ITER}_p{P}_q{Q}_w{WINDOW}.txt"
model_bip.wv.save_word2vec_format(str(emb_path))

print("Saved embeddings to:", emb_path)
print("Number of embeddings saved:", len(model_bip.wv))
print("Embedding dimension:", model_bip.vector_size)

# -----------------------------------------------------------
# C.0 — Evidence: nearest neighbours (reuse your canonical probes + helper)
# -----------------------------------------------------------

print_neighbours(
    model_bip,
    probe_ingredients_canonical,
    topn=10,
    title=f"{ITER} neighbours — ingredient projection from bipartite graph (deg-normalised)"
)


Loaded bipartite edges from: ../data/processed/ingredient_recipe_bipartite_edges.csv
Rows (edges): 1522980
Columns: ['node_u', 'node_v', 'weight']


,node_u,node_v,weight
0,recipe::137739,honey,1
1,recipe::137739,mexican,1
2,recipe::137739,mixed spice,1
3,recipe::137739,winter squash,1
4,recipe::31490,cheese,1



Bipartite graph built.
Recipe nodes: 219890
Ingredient nodes: 2698
Total nodes: 222588
Edges: 1522980

2-step ingredient projection built.
Unique ingredient-ingredient edges: 419913

Top 10 edges after bipartite-degree normalisation:


,ingredient_a,ingredient_b,count,deg_a,deg_b,weight
0,corn chex,wheat chex,37,90.0,50.0,0.551563
1,egg,flour,27623,51352.0,49386.0,0.548517
2,rice chex,wheat chex,40,107.0,50.0,0.546869
3,flour,powder,15622,49386.0,17454.0,0.532093
4,corn chex,rice chex,52,90.0,107.0,0.529895
5,nori,sushi rice,37,103.0,55.0,0.491589
6,flour,soda,12537,49386.0,14049.0,0.475958
7,egg,powder,13475,51352.0,17454.0,0.450093
8,powder,soda,6754,17454.0,14049.0,0.431311
9,tapioca flour,xanthan gum,85,147.0,276.0,0.421993



Saved projected ingredient edge list to: ../data/processed/ingredient_edges_from_bipartite_deg_norm.csv
Rows saved: 419913

Projected graph ready for Node2Vec.
Nodes: 2698
Edges: 419913


Generating walks (CPU: 2): 100%|██████████| 10/10 [00:54<00:00,  5.48s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou


Training complete: BIPPROJ_DEGNORM (p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_BIPPROJ_DEGNORM_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
BIPPROJ_DEGNORM neighbours — ingredient projection from bipartite graph (deg-normalised)
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. vanilla caramel                cosine=0.448
02. real vanilla                   cosine=0.411
03. vanilla chip                   cosine=0.410
04. milky way bar                  cosine=0.405
05. sifted flour                   cosine=0.401
06. milk chocolate candy bar       cosine=0.397
07. cris

In [1]:
# ===========================================================
# Z.0 — 10k BEST-QUALITY Bipartite Node2Vec (ingredient↔recipe)
# Direct training on bipartite graph + hub-mitigating weights
# ===========================================================

import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# -----------------------------
# Z.1 — Load / build 10k bipartite edges
# -----------------------------
# Preferred: a pre-built 10k bipartite edge list
BIP_10K_PATH = Path("../data/processed/ingredient_recipe_bipartite_edges_10k.csv")

# Fallback: sample 10k recipes from the full bipartite edge list if the 10k file doesn't exist
BIP_FULL_PATH = Path("../data/processed/ingredient_recipe_bipartite_edges.csv")

if BIP_10K_PATH.exists():
    bip_df = pd.read_csv(BIP_10K_PATH)
    print("Loaded 10k bipartite edges from:", BIP_10K_PATH)
else:
    if not BIP_FULL_PATH.exists():
        raise FileNotFoundError(
            "Neither 10k nor full bipartite edge list found.\n"
            f"Expected one of:\n- {BIP_10K_PATH}\n- {BIP_FULL_PATH}"
        )
    full_df = pd.read_csv(BIP_FULL_PATH)
    print("Loaded FULL bipartite edges from:", BIP_FULL_PATH)

    # Expect node_u is recipe::..., node_v is ingredient token
    if "node_u" not in full_df.columns or "node_v" not in full_df.columns:
        raise ValueError("Expected columns: node_u, node_v (and optional weight)")

    # Sample 10k distinct recipes, then keep their edges
    recipes = full_df["node_u"].astype(str).unique()
    if len(recipes) < 10_000:
        print(f"WARNING: only {len(recipes)} unique recipes available; using all of them.")
        chosen = set(recipes)
    else:
        rng = np.random.default_rng(42)
        chosen = set(rng.choice(recipes, size=10_000, replace=False))

    bip_df = full_df[full_df["node_u"].astype(str).isin(chosen)].copy()
    print("Constructed 10k subset from full file.")
    print("Edges kept:", len(bip_df))
    print("Unique recipes kept:", bip_df["node_u"].nunique())

# Standardize schema (ignore any existing 'weight' column — we will compute better weights)
bip_df = bip_df[["node_u", "node_v"]].copy()
bip_df["node_u"] = bip_df["node_u"].astype(str)
bip_df["node_v"] = bip_df["node_v"].astype(str)

assert bip_df["node_u"].str.startswith("recipe::").all(), "node_u must be recipe::..."
assert (~bip_df["node_v"].str.startswith("recipe::")).all(), "node_v must be ingredient tokens."

# -----------------------------
# Z.2 — Hub-mitigating edge weights: 1/deg(recipe)
# -----------------------------
# Idea: every recipe contributes a total weight of ~1 distributed across its ingredients.
# This prevents huge recipes (30+ ingredients) from dominating training.
recipe_deg = bip_df.groupby("node_u").size().to_dict()
bip_df["weight"] = bip_df["node_u"].map(recipe_deg).astype(float)
bip_df["weight"] = 1.0 / bip_df["weight"]

print("\n10k bipartite weight scheme:")
print("Recipe degree: min =", min(recipe_deg.values()), "max =", max(recipe_deg.values()))
print("Edge weight:   min =", bip_df["weight"].min(), "max =", bip_df["weight"].max())

# -----------------------------
# Z.3 — Build weighted bipartite graph
# -----------------------------
B = nx.Graph()
B.add_weighted_edges_from(
    bip_df[["node_u", "node_v", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

recipe_nodes = [n for n in B.nodes if str(n).startswith("recipe::")]
ingredient_nodes = [n for n in B.nodes if not str(n).startswith("recipe::")]

print("\nBipartite graph built (10k).")
print("Recipe nodes:", len(recipe_nodes))
print("Ingredient nodes:", len(ingredient_nodes))
print("Total nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

# -----------------------------
# Z.4 — Train Node2Vec directly on the bipartite graph
# -----------------------------
# “Role similarity” bias: higher q encourages exploring outward (DFS-ish),
# which helps “structural equivalence” (substitution-like) more than pure homophily.
DIM = 128
WALK_LEN = 80      # slightly longer helps in bipartite (alternating types)
NUM_WALKS = 15     # keep modest for 10k (faster, still decent)
WINDOW = 5         # smaller window reduces “everything becomes similar”
P = 1
Q = 4
SEED = 42

node2vec_bip_10k = Node2Vec(
    B,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",   # IMPORTANT: use hub-mitigating weights in transitions
    workers=4,
    seed=SEED
)

model_bip_10k = node2vec_bip_10k.fit(
    window=WINDOW,
    min_count=2,           # drop ultra-rare tokens from training dynamics
    batch_words=128,
    workers=2
)

print("\nTraining complete: BIPARTITE_10K (weighted by 1/deg(recipe)).")
print("Total vectors learned (recipes + ingredients):", len(model_bip_10k.wv))
print("Embedding dimension:", model_bip_10k.vector_size)

# -----------------------------
# Z.5 — Save ingredient-only embeddings
# -----------------------------
EMB_DIR = Path("../data/processed/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

ITER_TAG = f"BIPARTITE_10K_rdegW_p{P}_q{Q}_w{WINDOW}_l{WALK_LEN}_d{DIM}"
ing_only_path = EMB_DIR / f"ingredient_node2vec_{ITER_TAG}_ING_ONLY.txt"

ingredient_vocab = [t for t in model_bip_10k.wv.index_to_key if not str(t).startswith("recipe::")]
dim = model_bip_10k.vector_size

with open(ing_only_path, "w", encoding="utf-8") as f:
    f.write(f"{len(ingredient_vocab)} {dim}\n")
    for token in ingredient_vocab:
        vec = model_bip_10k.wv[token]
        f.write(token + " " + " ".join(f"{x:.6f}" for x in vec) + "\n")

print("Saved ingredient-only embeddings to:", ing_only_path)
print("Ingredient-only vectors:", len(ingredient_vocab))

# -----------------------------
# Z.6 — Evidence: neighbours for your canonical probes
# -----------------------------
def print_neighbours(model, probe_list, topn=10, title=""):
    vocab = set(model.wv.key_to_index.keys())
    present = [x for x in probe_list if x in vocab]
    missing = [x for x in probe_list if x not in vocab]

    if title:
        print("\n" + "#" * 70)
        print(title)
        print("#" * 70)

    print("Present probes:", present)
    print("Missing probes:", missing)

    for ing in present:
        print("\n" + "=" * 60)
        print(f"Probe: {ing}")
        for i, (nbr, sim) in enumerate(model.wv.most_similar(ing, topn=topn), start=1):
            # Hide recipe nodes in neighbour list (we only care ingredient neighbours)
            if str(nbr).startswith("recipe::"):
                continue
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

# Reuse your canonical probe list if it exists, otherwise define it
try:
    probe_ingredients_canonical
except NameError:
    probe_ingredients_canonical = [
        "real butter", "pure olive oil", "canola oil",
        "milk", "yogurt", "cream",
        "flour", "bread flour",
        "cane sugar", "light-brown sugar", "honey",
        "garlic", "red onion",
        "chicken", "beef",
        "rice", "pasta"
    ]

print_neighbours(
    model_bip_10k,
    probe_ingredients_canonical,
    topn=20,
    title="BIPARTITE 10K neighbours (ingredient-only view; recipe hubs downweighted)"
)


/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded FULL bipartite edges from: ../data/processed/ingredient_recipe_bipartite_edges.csv
Constructed 10k subset from full file.
Edges kept: 69367
Unique recipes kept: 10000

10k bipartite weight scheme:
Recipe degree: min = 3 max = 24
Edge weight:   min = 0.041666666666666664 max = 0.3333333333333333

Bipartite graph built (10k).
Recipe nodes: 10000
Ingredient nodes: 2425
Total nodes: 12425
Edges: 69367


Generating walks (CPU: 3): 100%|██████████| 4/4 [00:44<00:00, 11.07s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_


Training complete: BIPARTITE_10K (weighted by 1/deg(recipe)).
Total vectors learned (recipes + ingredients): 12425
Embedding dimension: 128
Saved ingredient-only embeddings to: ../data/processed/embeddings/ingredient_node2vec_BIPARTITE_10K_rdegW_p1_q4_w5_l80_d128_ING_ONLY.txt
Ingredient-only vectors: 2425

######################################################################
BIPARTITE 10K neighbours (ingredient-only view; recipe hubs downweighted)
######################################################################
Present probes: ['real butter', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: ['pure olive oil']

Probe: real butter
02. real vanilla extract           cosine=0.754
07. bailey irish cream             cosine=0.614
08. caramel ice cream topping      cosine=0.562
16. chai tea                       cosine=0.529
17. bread dough             

In [1]:
# ===========================================================
# K.0 — (RESEARCH-RECOMMENDED) Node2Vec on the LOSSLESS
# INGREDIENT–RECIPE BIPARTITE GRAPH (NO PROJECTION)
#
# Key idea:
# - Train Node2Vec directly on the bipartite graph B (ingredient <-> recipe edges).
# - Walks naturally alternate: ingredient -> recipe -> ingredient -> ...
# - Keep edges UNWEIGHTED (weight=1, and we do NOT pass weight_key).
# - For substitution, we ONLY evaluate + save embeddings for ingredient nodes.
# ===========================================================

import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# -----------------------------------------------------------
# K.1 — Load the frozen bipartite edge list (read-only)
# -----------------------------------------------------------

BIP_EDGES_PATH = "../data/processed/ingredient_recipe_bipartite_edges.csv"
bip_df = pd.read_csv(BIP_EDGES_PATH)

print("Loaded bipartite edges from:", BIP_EDGES_PATH)
print("Rows (edges):", len(bip_df))
print("Columns:", list(bip_df.columns))
display(bip_df.head())

# Safety checks (match the bipartite design)
assert set(bip_df.columns) == {"node_u", "node_v", "weight"}, "Unexpected columns."
assert (bip_df["node_u"].astype(str).str.startswith("recipe::")).all(), "node_u must be recipe::..."
assert (~bip_df["node_v"].astype(str).str.startswith("recipe::")).all(), "node_v must be ingredient tokens."
assert (bip_df["weight"] > 0).all(), "Found non-positive weights."
assert (bip_df["weight"] == 1).all(), "Expected all bipartite weights to be 1."

# -----------------------------------------------------------
# K.2 — Build the bipartite graph B (UNWEIGHTED)
# -----------------------------------------------------------

B = nx.Graph()
B.add_edges_from(bip_df[["node_u", "node_v"]].itertuples(index=False, name=None))

recipe_nodes = [n for n in B.nodes if str(n).startswith("recipe::")]
ingredient_nodes = [n for n in B.nodes if not str(n).startswith("recipe::")]

print("\nBipartite graph built (UNWEIGHTED).")
print("Recipe nodes:", len(recipe_nodes))
print("Ingredient nodes:", len(ingredient_nodes))
print("Total nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

# -----------------------------------------------------------
# K.3 — Train Node2Vec on B (research-style, unweighted)
# ONLY CHANGE vs previous slow run: workers increased to 8
# -----------------------------------------------------------

ITER = "BIPARTITE_DIRECT_UNWEIGHTED"
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
WINDOW = 5
P = 1
Q = 4
SEED = 42

node2vec_bip_direct = Node2Vec(
    B,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: unweighted (no weight_key)
    workers=8,     # <-- ONLY intentional change (speed-up)
    seed=SEED
)

model_bip_direct = node2vec_bip_direct.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=8     # <-- match CPU usage during training
)

print(f"\nTraining complete: {ITER} (UNWEIGHTED, p={P}, q={Q}, window={WINDOW}).")
print("Total embeddings learned (ingredients + recipes):", len(model_bip_direct.wv))
print("Embedding dimension:", model_bip_direct.vector_size)

# -----------------------------------------------------------
# K.4 — Save embeddings
# -----------------------------------------------------------

EMB_DIR = Path("../data/processed/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

full_path = EMB_DIR / f"node2vec_{ITER}_FULL_p{P}_q{Q}_w{WINDOW}_l{WALK_LEN}.txt"
ing_only_path = EMB_DIR / f"node2vec_{ITER}_INGREDIENTS_ONLY_p{P}_q{Q}_w{WINDOW}_l{WALK_LEN}.txt"

model_bip_direct.wv.save_word2vec_format(str(full_path))
print("Saved FULL embeddings to:", full_path)

dim = model_bip_direct.vector_size
ingredient_vocab = [n for n in model_bip_direct.wv.index_to_key if not str(n).startswith("recipe::")]

with open(ing_only_path, "w", encoding="utf-8") as f:
    f.write(f"{len(ingredient_vocab)} {dim}\n")
    for token in ingredient_vocab:
        vec = model_bip_direct.wv[token]
        f.write(token + " " + " ".join(f"{x:.6f}" for x in vec) + "\n")

print("Saved INGREDIENT-ONLY embeddings to:", ing_only_path)
print("Ingredient-only vectors:", len(ingredient_vocab))


/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded bipartite edges from: ../data/processed/ingredient_recipe_bipartite_edges.csv
Rows (edges): 1522980
Columns: ['node_u', 'node_v', 'weight']


,node_u,node_v,weight
0,recipe::137739,honey,1
1,recipe::137739,mexican,1
2,recipe::137739,mixed spice,1
3,recipe::137739,winter squash,1
4,recipe::31490,cheese,1



Bipartite graph built (UNWEIGHTED).
Recipe nodes: 219890
Ingredient nodes: 2698
Total nodes: 222588
Edges: 1522980


Computing transition probabilities: 100%|██████████| 222588/222588 [4:05:41<00:00, 15.10it/s]     


: 